In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [5]:
import enum
import instructor
from pydantic import BaseModel
import os

# Configure Tracing

In [6]:
from langfuse.decorators import langfuse_context, observe
from langfuse.openai import OpenAI

langfuse_context.configure(
    secret_key=os.getenv('LANGFUSE_SECRET_KEY'),
    public_key=os.getenv('LANGFUSE_PUBLIC_KEY'),
    host=os.getenv('LANGFUSE_INTERNAL_HOST'),
    max_retries=1,  # Default: 3
    timeout=5,  # Default: 20
)

# Implement a GenAI based classification method


In [17]:
from llmetrics.langfuse_adaptor import SensitivityConsistencyMetrics


class Labels(str, enum.Enum):
    """Enumeration for single-label
    text classification."""
    NUM = "Number",
    DESC = "Description",
    ENTY = "Entity",
    ABBR = "Abbreviation",
    LOC = "Location",
    HUM = "Person",


class SinglePrediction(BaseModel):
    """
    Class for a single class label prediction.
    """
    class_label: Labels


@observe(name="classify")
def classify(prompt: str,  user_input: str, model: str = "gpt-4o-mini") -> Labels | None:
    labels_list = ['Number', 'Location', 'Person', 'Description', 'Entity', 'Abbreviation'] 
      
    client = instructor.from_openai(OpenAI())
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": f"{prompt} \n {user_input}"}],
            response_model=SinglePrediction,  
            max_retries=3,
        )
    except Exception as e:
        print(
            f"Error {type(e).__name__} exception: {str(e)}"
        )
        return None
    
    # Append Sensitivity & Consistency Metadata to Trace
    # parameters: 
    # - prompt: promt string used to query the llm
    # - response_model: # TODO make this optional 
    # - user_input: input to be classified
    # - response.class_label: parsed output 
    # - labels_list: list of possible labels
    # - model: llm model 
    # - langfuse_context: langfuse context object
    
    SensitivityConsistencyMetrics.append_sensitivity_metadata_to_trace(prompt, 
                                                           SinglePrediction, 
                                                           user_input, 
                                                           response.class_label, 
                                                           labels_list,
                                                           model, 
                                                           langfuse_context)
       
    return response

In [18]:
test_inputs = [
    'How far is it from Denver to Aspen ?',
    'What county is Modesto , California in ?',
    'Who was Galileo ?',
    'What is an atom ?',
    'When did Hawaii become a state ?',
    'How tall is the Sears Building ?',
    'George Bush purchased a small interest in which baseball team ?',
    "What is Australia 's national flower ?",
    'Why does the moon turn orange ?',
    'What is autism ?'
]


## Test a first prompt implementation 

In [20]:
prompt_1 = 'Classify the questions based on whether their answer type is a Number, Location, Person, Description, Entity, or Abbreviation.'
for t in test_inputs:
    print(t)
    result = classify(prompt_1,t)
    print(result)

How far is it from Denver to Aspen ?
class_label=<Labels.NUM: 'Number'>
What county is Modesto , California in ?
class_label=<Labels.LOC: 'Location'>
Who was Galileo ?
class_label=<Labels.HUM: 'Person'>
What is an atom ?
class_label=<Labels.DESC: 'Description'>
When did Hawaii become a state ?
class_label=<Labels.LOC: 'Location'>
How tall is the Sears Building ?
class_label=<Labels.NUM: 'Number'>
George Bush purchased a small interest in which baseball team ?
class_label=<Labels.ENTY: 'Entity'>
What is Australia 's national flower ?
class_label=<Labels.LOC: 'Location'>
Why does the moon turn orange ?
class_label=<Labels.DESC: 'Description'>
What is autism ?
class_label=<Labels.DESC: 'Description'>


## Test a second prompt implementation 

In [21]:
prompt_2 = 'Determine the answer type for each question: Number, Location, Person, Description, Entity, or Abbreviation.'

for t in test_inputs:
    print(t)
    result = classify(prompt_2,t)
    print(result)


How far is it from Denver to Aspen ?
class_label=<Labels.NUM: 'Number'>
What county is Modesto , California in ?
class_label=<Labels.LOC: 'Location'>
Who was Galileo ?
class_label=<Labels.HUM: 'Person'>
What is an atom ?
class_label=<Labels.DESC: 'Description'>
When did Hawaii become a state ?
class_label=<Labels.LOC: 'Location'>
How tall is the Sears Building ?
class_label=<Labels.NUM: 'Number'>
George Bush purchased a small interest in which baseball team ?
class_label=<Labels.LOC: 'Location'>
What is Australia 's national flower ?
class_label=<Labels.DESC: 'Description'>
Why does the moon turn orange ?
class_label=<Labels.DESC: 'Description'>
What is autism ?
class_label=<Labels.DESC: 'Description'>


## ... Test a nth prompt implementation 

In [22]:
prompt_n = 'Categorize the questions based on whether their answers are classified as a Number, Location, Person, Description, Entity, or Abbreviation.'

for t in test_inputs:
    print(t)
    result = classify(prompt_n,t)
    print(result)

How far is it from Denver to Aspen ?
class_label=<Labels.LOC: 'Location'>
What county is Modesto , California in ?
class_label=<Labels.LOC: 'Location'>
Who was Galileo ?
class_label=<Labels.HUM: 'Person'>
What is an atom ?
class_label=<Labels.DESC: 'Description'>
When did Hawaii become a state ?
class_label=<Labels.LOC: 'Location'>
How tall is the Sears Building ?
class_label=<Labels.NUM: 'Number'>
George Bush purchased a small interest in which baseball team ?
class_label=<Labels.ENTY: 'Entity'>
What is Australia 's national flower ?
class_label=<Labels.LOC: 'Location'>
Why does the moon turn orange ?
class_label=<Labels.DESC: 'Description'>
What is autism ?
class_label=<Labels.DESC: 'Description'>


## Retrieve Sensitivity Metrics

In [33]:
from llmetrics.langfuse_adaptor import SensitivityConsistencyMetrics

os.environ['NO_PROXY'] = 'localhost' # TODO: if needed
metrics = SensitivityConsistencyMetrics(trace_name='classify')
traces = metrics.fetch_traces()
sensitivity = metrics.compute_sensitivity(traces)
sensitivity

Retrieved 41 traces for classify from 2025-03-26 23:59:00 to 2025-03-27 23:59:00.


,input_tag,entropy,distribution,expected_output
0,input-21f0b2df5a63bb490e2d2e349d3d37a88f86eccc...,0.000138,"[0.0, 0.0, 0.0, 0.0, 1.0, 0.0]",None
1,input-390b4d16b3b9d9dd19ea30954007043dbff34baa...,0.135324,"[0.0, 0.0, 0.75, 0.25, 0.0, 0.0]",None
2,input-4d89eb3dedc0a3f61b947a09b03084f22591cc46...,0.000138,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0]",None
3,input-5902fbfe31c9915dd8a6c7502cbaef5d82f94fbc...,0.000138,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]",None
4,input-60af75e02b0c8903b336e83a503106d5139b3c37...,0.000138,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]",None
5,input-8612975439a932bf3020a97aa2dd4046b4bd19c2...,0.000138,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0]",None
6,input-9d8c189fa6359aae6c1accfa2f287847d2ad1ae6...,0.000138,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0]",None
7,input-ba596ff6c54b2c49cb2008d6f659c358be9aa6df...,0.000138,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0]",None
8,input-dafe4c3ebf85739b2df222656f233ed8e34164ac...,0.135324,"[0.0, 0.25, 0.0, 0.75, 0.0, 0.0]",None
9,input-e40035b9beb5d2760a9c660c0e2be8e4cb0163d8...,0.135324,"[0.0, 0.0, 0.0, 0.25, 0.75, 0.0]",None


# Compute Consistency 

## Create a Dataset


In [42]:
from langfuse import Langfuse

langfuse = Langfuse(
            secret_key=os.getenv('LANGFUSE_SECRET_KEY'),
            public_key=os.getenv('LANGFUSE_PUBLIC_KEY'),
            host=os.getenv('LANGFUSE_INTERNAL_HOST'),
        )

In [45]:
langfuse.create_dataset(
    name="test_dataset",
)

Dataset(id='cm8rig56d00k4emofr6etilss', name='test_dataset', description=None, metadata=None, project_id='cm88ld4kq0006emof44aofxxo', created_at=datetime.datetime(2025, 3, 27, 15, 30, 41, 846000, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2025, 3, 27, 15, 30, 41, 846000, tzinfo=datetime.timezone.utc))

## Add items to the dataset

In [34]:
test_inputs

['How far is it from Denver to Aspen ?',
 'What county is Modesto , California in ?',
 'Who was Galileo ?',
 'What is an atom ?',
 'When did Hawaii become a state ?',
 'How tall is the Sears Building ?',
 'George Bush purchased a small interest in which baseball team ?',
 "What is Australia 's national flower ?",
 'Why does the moon turn orange ?',
 'What is autism ?']

In [40]:
test_output = ['Number',
 'Location',
 'Person',
 'Description',
 'Number',
 'Number',
 'Person',
 'Entity',
 'Description',
 'Description']

In [46]:
for t,o in zip(test_inputs, test_output):
    langfuse.create_dataset_item(
        dataset_name="test_dataset",
        input=t,
        expected_output=o,
    )

In [52]:
consistency, consistency_matrix = metrics.compute_consistency(traces,dataset_name="test_dataset")

In [53]:
consistency

{'Description': np.float64(1.0),
 'Entity': nan,
 'Location': nan,
 'Number': np.float64(0.3333333333333333),
 'Person': np.float64(0.0)}

In [54]:
consistency_matrix

{'Description': array([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]]),
 'Entity': array([[1.]]),
 'Location': array([[1.]]),
 'Number': array([[1.  , 0.  , 0.75],
        [0.  , 1.  , 0.25],
        [0.75, 0.25, 1.  ]]),
 'Person': array([[1., 0.],
        [0., 1.]])}